In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
from spacy.lang.en import English
from datasets import load_dataset
from collections import Counter
import itertools

nlp = English()

def tokenize(text):
    return [tok.text for tok in nlp(text)]

ds = load_dataset("fancyzhx/ag_news")
ds = ds.map(lambda x: {"normalized_text": x["text"].lower()})
token_counts = Counter(
    itertools.chain.from_iterable(tokenize(text) for text in ds["train"]["normalized_text"]))
vocab = {"<pad>": 0, "<unk>": 1}
for token, count in token_counts.items():
    if count >= 3:
        vocab[token] = len(vocab)

def encode(sample):
    doc = nlp(sample['normalized_text'])
    ids = [vocab.get(token.text, 1) for token in doc]
    return {"input_ids": ids}

ds = ds.map(encode)

In [ ]:
print(ds)

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

def collate_fn(batch):
    lengths = torch.tensor([len(sample["input_ids"]) for sample in batch])
    seqs = [torch.tensor(sample["input_ids"]) for sample in batch]
    padded = pad_sequence(seqs)
    labels = torch.tensor([sample["label"] for sample in batch])
    return padded, lengths, labels

train_loader = DataLoader(ds["train"], batch_size=64, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(ds["test"], batch_size=64, shuffle=False, collate_fn=collate_fn)


In [ ]:
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence

class Net(nn.Module):
    def __init__(self, num_embeddings, output_size):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=num_embeddings, embedding_dim=100, padding_idx=0)
        self.lstm = nn.LSTM(input_size=100, hidden_size=128)
        self.fc = nn.Linear(128, output_size)
    
    def forward(self, input, lengths):
        x = self.embedding(input)
        x = pack_padded_sequence(x, lengths, enforce_sorted=False)
        output, (h_n, c_n) = self.lstm(x)
        x = self.fc(h_n[-1])
        return x

net = Net(num_embeddings=len(vocab), output_size=4)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=1e-3)

In [ ]:
for epoch in range(5):
    running_loss = 0.0
    count = 0
    for i, data in enumerate(train_loader):
        padded, lengths, labels = data

        optimizer.zero_grad()

        outputs = net(padded, lengths)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        count += 1

        if i % 100 == 0:
            print(f'[{epoch + 1}, {i}] loss: {running_loss / count:.3f}')
            running_loss = 0.0
            count = 0

print('Finished training')

In [ ]:
correct = 0
total = 0

with torch.no_grad():
    for data in test_loader:
        padded, lengths, labels = data

        output = net(padded, lengths)
        _, predicted = torch.max(output, dim=1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the test data: {100 * correct // total} %')